# 09 · Conditional Local Structure After Small and Large Gaps

This notebook studies **conditional local structure** in short gap windows.

The question is:

> how does the next local pattern behave after an unusually small gap or an unusually large gap?

We compare three ensembles:

1. zeta-zero gaps
2. Poisson / exponential control gaps
3. finite GUE-matrix bulk gaps

## Why this matters

Earlier notebooks showed that zeta and GUE look similar in:

- spacing distributions
- ratio statistics
- pair correlation
- 4-gap and 5-gap window descriptors

Here we go one step further:
we condition on a local event and ask whether the *following short pattern* differs systematically.

This is still an exploratory numerical notebook, but it is a natural next step beyond unconditional local statistics.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Data sources

In [ ]:
# Zeta gaps
N = 700
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=rng)

len(zeta_gaps), len(poisson_gaps), len(gue_gaps)

## Normalize by the global mean for thresholding

To define "small" and "large" gaps in a comparable way, we first divide each gap sequence by its own mean.
This is only for choosing the conditional events.

In [ ]:
zeta_g = zeta_gaps / zeta_gaps.mean()
poisson_g = poisson_gaps / poisson_gaps.mean()
gue_g = gue_gaps / gue_gaps.mean()

zeta_g.mean(), poisson_g.mean(), gue_g.mean()

## Define conditional events

We use simple quantile thresholds:

- **small gap** = bottom 20%
- **large gap** = top 20%

Then we examine the next local 4-gap or 5-gap window *starting at that event*.

In [ ]:
def quantile_thresholds(gaps, q_low=0.2, q_high=0.8):
    return np.quantile(gaps, q_low), np.quantile(gaps, q_high)

zeta_low, zeta_high = quantile_thresholds(zeta_g)
poisson_low, poisson_high = quantile_thresholds(poisson_g)
gue_low, gue_high = quantile_thresholds(gue_g)

zeta_low, zeta_high, poisson_low, poisson_high, gue_low, gue_high

## Extract conditional windows

For each index \(n\):

- if \(g_n\) is unusually small, collect the next window
- if \(g_n\) is unusually large, collect the next window

Each window is normalized by its own mean so we compare shape rather than scale.

In [ ]:
def conditional_windows(gaps, low_thr, high_thr, k):
    small = []
    large = []
    for i in range(len(gaps) - k + 1):
        w = gaps[i:i+k]
        w_norm = w / w.mean()
        if gaps[i] <= low_thr:
            small.append(w_norm)
        if gaps[i] >= high_thr:
            large.append(w_norm)
    return np.array(small), np.array(large)

zeta_small4, zeta_large4 = conditional_windows(zeta_g, zeta_low, zeta_high, 4)
poisson_small4, poisson_large4 = conditional_windows(poisson_g, poisson_low, poisson_high, 4)
gue_small4, gue_large4 = conditional_windows(gue_g, gue_low, gue_high, 4)

zeta_small5, zeta_large5 = conditional_windows(zeta_g, zeta_low, zeta_high, 5)
poisson_small5, poisson_large5 = conditional_windows(poisson_g, poisson_low, poisson_high, 5)
gue_small5, gue_large5 = conditional_windows(gue_g, gue_low, gue_high, 5)

zeta_small4.shape, zeta_large4.shape, gue_small5.shape

## Mean conditional 4-gap profiles

This is the first main comparison:
- mean profile after a small gap
- mean profile after a large gap

In [ ]:
def mean_profile(windows):
    return windows.mean(axis=0)

x4 = np.arange(1, 5)

plt.figure(figsize=(8.8, 5.0))
plt.plot(x4, mean_profile(zeta_small4), marker="o", label="zeta after small")
plt.plot(x4, mean_profile(zeta_large4), marker="o", label="zeta after large")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 4-gap window")
plt.ylabel("mean normalized gap")
plt.title("Zeta conditional 4-gap profiles")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
plt.plot(x4, mean_profile(poisson_small4), marker="o", label="Poisson after small")
plt.plot(x4, mean_profile(poisson_large4), marker="o", label="Poisson after large")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 4-gap window")
plt.ylabel("mean normalized gap")
plt.title("Poisson conditional 4-gap profiles")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
plt.plot(x4, mean_profile(gue_small4), marker="o", label="GUE after small")
plt.plot(x4, mean_profile(gue_large4), marker="o", label="GUE after large")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 4-gap window")
plt.ylabel("mean normalized gap")
plt.title("GUE conditional 4-gap profiles")
plt.legend()
plt.show()

## Mean conditional 5-gap profiles

In [ ]:
x5 = np.arange(1, 6)

plt.figure(figsize=(8.8, 5.0))
plt.plot(x5, mean_profile(zeta_small5), marker="o", label="zeta after small")
plt.plot(x5, mean_profile(zeta_large5), marker="o", label="zeta after large")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 5-gap window")
plt.ylabel("mean normalized gap")
plt.title("Zeta conditional 5-gap profiles")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
plt.plot(x5, mean_profile(poisson_small5), marker="o", label="Poisson after small")
plt.plot(x5, mean_profile(poisson_large5), marker="o", label="Poisson after large")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 5-gap window")
plt.ylabel("mean normalized gap")
plt.title("Poisson conditional 5-gap profiles")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
plt.plot(x5, mean_profile(gue_small5), marker="o", label="GUE after small")
plt.plot(x5, mean_profile(gue_large5), marker="o", label="GUE after large")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 5-gap window")
plt.ylabel("mean normalized gap")
plt.title("GUE conditional 5-gap profiles")
plt.legend()
plt.show()

## Conditional descriptor helpers

We reuse the earlier higher-order descriptors:
- unevenness
- reversal symmetry
- entropy

In [ ]:
def unevenness(windows):
    return windows.max(axis=1) - windows.min(axis=1)

def reversal_symmetry_score(windows):
    rev = windows[:, ::-1]
    return np.mean(np.abs(windows - rev), axis=1)

def window_entropy(windows):
    eps = 1e-12
    p = windows / windows.sum(axis=1, keepdims=True)
    return -np.sum(p * np.log(p + eps), axis=1)

## 4-gap conditional unevenness

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.0, 32)
plt.hist(unevenness(zeta_small4), bins=bins, density=True, alpha=0.5, label="zeta after small")
plt.hist(unevenness(zeta_large4), bins=bins, density=True, alpha=0.5, label="zeta after large")
plt.xlabel("unevenness")
plt.ylabel("density")
plt.title("Zeta 4-gap conditional unevenness")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.0, 32)
plt.hist(unevenness(gue_small4), bins=bins, density=True, alpha=0.5, label="GUE after small")
plt.hist(unevenness(gue_large4), bins=bins, density=True, alpha=0.5, label="GUE after large")
plt.xlabel("unevenness")
plt.ylabel("density")
plt.title("GUE 4-gap conditional unevenness")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.0, 32)
plt.hist(unevenness(poisson_small4), bins=bins, density=True, alpha=0.5, label="Poisson after small")
plt.hist(unevenness(poisson_large4), bins=bins, density=True, alpha=0.5, label="Poisson after large")
plt.xlabel("unevenness")
plt.ylabel("density")
plt.title("Poisson 4-gap conditional unevenness")
plt.legend()
plt.show()

## 5-gap conditional entropy

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0.9, np.log(5), 30)
plt.hist(window_entropy(zeta_small5), bins=bins, density=True, alpha=0.5, label="zeta after small")
plt.hist(window_entropy(zeta_large5), bins=bins, density=True, alpha=0.5, label="zeta after large")
plt.xlabel("window entropy")
plt.ylabel("density")
plt.title("Zeta 5-gap conditional entropy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0.9, np.log(5), 30)
plt.hist(window_entropy(gue_small5), bins=bins, density=True, alpha=0.5, label="GUE after small")
plt.hist(window_entropy(gue_large5), bins=bins, density=True, alpha=0.5, label="GUE after large")
plt.xlabel("window entropy")
plt.ylabel("density")
plt.title("GUE 5-gap conditional entropy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0.9, np.log(5), 30)
plt.hist(window_entropy(poisson_small5), bins=bins, density=True, alpha=0.5, label="Poisson after small")
plt.hist(window_entropy(poisson_large5), bins=bins, density=True, alpha=0.5, label="Poisson after large")
plt.xlabel("window entropy")
plt.ylabel("density")
plt.title("Poisson 5-gap conditional entropy")
plt.legend()
plt.show()

## Conditional symmetry summary

Compare the mean reversal-symmetry score after small vs large gaps.

In [ ]:
conditional_summary = {
    "zeta_4gap_symmetry": {
        "after_small": float(reversal_symmetry_score(zeta_small4).mean()),
        "after_large": float(reversal_symmetry_score(zeta_large4).mean()),
    },
    "poisson_4gap_symmetry": {
        "after_small": float(reversal_symmetry_score(poisson_small4).mean()),
        "after_large": float(reversal_symmetry_score(poisson_large4).mean()),
    },
    "gue_4gap_symmetry": {
        "after_small": float(reversal_symmetry_score(gue_small4).mean()),
        "after_large": float(reversal_symmetry_score(gue_large4).mean()),
    },
    "zeta_5gap_entropy": {
        "after_small": float(window_entropy(zeta_small5).mean()),
        "after_large": float(window_entropy(zeta_large5).mean()),
    },
    "poisson_5gap_entropy": {
        "after_small": float(window_entropy(poisson_small5).mean()),
        "after_large": float(window_entropy(poisson_large5).mean()),
    },
    "gue_5gap_entropy": {
        "after_small": float(window_entropy(gue_small5).mean()),
        "after_large": float(window_entropy(gue_large5).mean()),
    },
}
conditional_summary

## Difference profiles: after large minus after small

This makes the conditional effect easier to see directly.

In [ ]:
plt.figure(figsize=(8.8, 5.0))
plt.plot(x5, mean_profile(zeta_large5) - mean_profile(zeta_small5), marker="o", label="zeta")
plt.plot(x5, mean_profile(poisson_large5) - mean_profile(poisson_small5), marker="o", label="Poisson")
plt.plot(x5, mean_profile(gue_large5) - mean_profile(gue_small5), marker="o", label="GUE")
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.xlabel("position in 5-gap window")
plt.ylabel("large-minus-small profile")
plt.title("Conditional 5-gap difference profile")
plt.legend()
plt.show()

## Notes

- Conditioning on an unusually small or large gap is a simple way to probe whether short local structure has a memory effect.
- If zeta and GUE show similar conditional shifts while Poisson behaves differently, that supports the idea that higher-order local structure is tied to spectral correlations rather than independent randomness.
- This notebook uses straightforward quantile thresholds and local mean normalization; more refined versions could:
  - vary the thresholds
  - condition on middle vs extreme quantiles
  - use longer windows
  - compare conditional descriptor vectors directly

## Next directions

A natural next notebook could do one of these:

1. embed the 4-gap / 5-gap descriptor vectors into 2D and compare zeta / Poisson / GUE clusters  
2. test several quantile thresholds for conditional stability  
3. combine the normalized pair-correlation benchmark with conditional window statistics  
4. build a simple classifier on descriptor vectors to distinguish the three ensembles